![image.png](https://i.imgur.com/a3uAqnb.png)

# Preference Tuning With DPO: Short Answers Without Asking

- **Dataset**: [Natural Questions (open)](https://huggingface.co/datasets/google-research-datasets/nq_open), short factual questions with reference answers, and a look at [UltraFeedback](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized), a public set of preference pairs
- **Task**: teach a chat model to answer briefly by default, from pairs of a better and a worse answer
- **Model**: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), trained with a LoRA adapter
- **Runtime**: Colab T4 GPU, about 10 minutes of compute

In this lab, we will:

- look at real preference pairs
- measure how long and how correct the model's answers are
- build our own pairs: the model answers each question twice, once asked to be concise
- train with TRL's `DPOTrainer` and follow its logged rewards
- measure again, and compare with simply asking for concise answers in every prompt
- look inside DPO at the log-probabilities of one pair, and compare with SFT on the chosen answers

## DPO in brief

Supervised fine-tuning copies one answer per prompt. Preference tuning learns from two answers to the same prompt, one marked better (chosen, $y_w$) and one worse (rejected, $y_l$). DPO trains the model $\pi$ to raise the chosen answer's probability and lower the rejected one's, each measured against a frozen copy $\pi_{\text{ref}}$:

$$\text{margin} = \beta \left[\log \frac{\pi(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \log \frac{\pi(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right]$$

Where:
- $x$ is the prompt, and each log ratio says how much more likely the answer has become than in the frozen copy
- the loss pushes the margin up, through $-\log \sigma(\text{margin})$
- $\beta$ (0.1 by default) sets how far the model may move from the frozen copy

No reward model and no sampling during training are needed: only the pairs.

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. The next cell installs TRL.

In [ ]:
%pip install -q trl

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

hf_logging.set_verbosity_error()   # hide informational warnings from model loading and training

Every size and model name lives in the next cell. `N_TRAIN` sets how many questions become preference pairs, and `N_TEST` how many questions are scored.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-0.6B"
N_TRAIN = 300            # questions the model answers twice to make pairs
N_TEST = 50              # questions scored before and after
BETA = 0.1
LEARNING_RATE = 1e-5     # higher rates can push the model to answers too short to be right
EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM = 2
MAX_NEW_TOKENS = 200     # long enough to see a wordy answer
CONCISE = " Be concise."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8   # a T4 (7.5) has no bfloat16
FP16 = DEVICE == "cuda" and not BF16
DTYPE = torch.bfloat16 if BF16 else torch.float16 if FP16 else torch.float32
print(f"PyTorch {torch.__version__} | device: {DEVICE} | weights in {DTYPE}")

## 1. What preference data looks like

UltraFeedback collected answers from several models to the same prompts and had a strong model rate them. The binarized version keeps, for each prompt, a higher-rated answer and a lower-rated one.

In [ ]:
UF = "hf://datasets/HuggingFaceH4/ultrafeedback_binarized/data/test_prefs-00000-of-00001.parquet"   # 7 MB
uf = load_dataset("parquet", data_files=UF, split="train[:3]")
for row in uf:
    print("PROMPT:  ", row["prompt"][:200])
    print("CHOSEN:  ", row["chosen"][-1]["content"][:200], f"(score {row['score_chosen']})")
    print("REJECTED:", row["rejected"][-1]["content"][:200], f"(score {row['score_rejected']})\n")

## 2. The model and the scorer

Each question is sent as a plain user message, with no system message. We record how many words the answer has, how many answers have at most 15 words, and how many contain one of the reference answers (a rough check of correctness).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)

nq_train = load_dataset("google-research-datasets/nq_open", split="train").shuffle(seed=SEED).select(range(N_TRAIN))
nq_test = load_dataset("google-research-datasets/nq_open", split="validation").select(range(N_TEST))


def ask(question, suffix=""):
    text = question[0].upper() + question[1:] + "?" + suffix
    return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)


@torch.no_grad()
def generate(m, prompts, max_new_tokens=MAX_NEW_TOKENS, batch_size=16):
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]


def score(m, suffix="", name=""):
    outs = generate(m, [ask(q, suffix) for q in nq_test["question"]])
    words = [len(o.split()) for o in outs]
    hits = [any(a.lower() in o.lower() for a in refs) for o, refs in zip(outs, nq_test["answer"])]
    return pd.Series({"mean words": np.mean(words), "at most 15 words": sum(w <= 15 for w in words),
                      "contains reference": sum(hits)}, name=name), outs


before, before_outs = score(model, name="before")
asked, asked_outs = score(model, CONCISE, name="before, prompt says 'Be concise.'")
pd.concat([before, asked], axis=1).T

In [ ]:
for q, a, b in list(zip(nq_test["question"], before_outs, asked_outs))[:3]:
    print(q, "\n  plain:  ", a[:200], "\n  concise:", b[:200], "\n")

## 3. Build the pairs

The model answers each training question twice: once with "Be concise." added (the chosen answer) and once without (the rejected answer). Both answers come from the model we train, which suits DPO: it compares their probabilities under that same model. Pairs where the plain answer is not clearly longer are dropped. The prompt of every pair is the plain question, so the model learns to be brief without being asked.

In [ ]:
start = time.time()
chosen = generate(model, [ask(q, CONCISE) for q in nq_train["question"]], max_new_tokens=80)
rejected = generate(model, [ask(q) for q in nq_train["question"]])
pairs = [{"prompt": ask(q), "chosen": c + "<|im_end|>", "rejected": r + "<|im_end|>"}
         for q, c, r in zip(nq_train["question"], chosen, rejected) if len(c.split()) + 10 < len(r.split())]
print(f"{len(pairs)} pairs kept out of {N_TRAIN} questions ({time.time() - start:.0f} s)")
print("chosen, mean words:  ", np.mean([len(p["chosen"].split()) for p in pairs]).round(1))
print("rejected, mean words:", np.mean([len(p["rejected"].split()) for p in pairs]).round(1))
print("\nexample\n", pairs[0]["chosen"][:200], "\n---\n", pairs[0]["rejected"][:300])

## 4. Train with DPO

With a LoRA adapter, no reference model is passed: the trainer uses the base model with the adapter turned off as the frozen copy. The learning rate is lower than for SFT, because DPO only adjusts a model that already answers well.

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
args = DPOConfig(output_dir="dpo-lora", beta=BETA, learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
                 per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM, logging_steps=2,
                 max_length=512, save_strategy="no", report_to="none", seed=SEED, fp16=FP16, bf16=BF16,
                 use_cpu=DEVICE == "cpu")
trainer = DPOTrainer(model=model, args=args, train_dataset=Dataset.from_list(pairs), processing_class=tokenizer,
                     peft_config=lora_config)
start = time.time()
trainer.train()
print(f"training took {time.time() - start:.0f} s")

The trainer logs, for each step, how often the chosen answer already scores above the rejected one (`rewards/accuracies`) and by how much (`rewards/margins`). Both should rise.

In [ ]:
logs = pd.DataFrame([h for h in trainer.state.log_history if "rewards/margins" in h])
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["loss", "rewards/accuracies", "rewards/margins"]):
    ax.plot(logs["step"], logs[col], marker="o")
    ax.set_title(col)
    ax.set_xlabel("step")
plt.tight_layout()
plt.show()

## 5. Measure again

The plain question, no instruction about length.

In [ ]:
tuned = trainer.model.eval()
after, after_outs = score(tuned, name="after DPO, plain question")
pd.concat([before, asked, after], axis=1).T

In [ ]:
for q, a, b in list(zip(nq_test["question"], before_outs, after_outs))[:3]:
    print(q, "\n  before:", a[:200], "\n  after: ", b[:200], "\n")

Asking for concise answers costs a few tokens on every request and must be remembered by every caller; the tuned model does it by default. Check the correctness column as well: shorter is only better if the answer survives.

## 6. Inside DPO: one pair, before and after

DPO's margin is built from the log-probability of each whole answer, $\log \pi(y \mid x)$: the sum of the log-probabilities of its tokens. The next cell computes it for a few training pairs with the adapter off (the frozen copy) and on (the trained model). The log ratio is the difference between the two.

In [ ]:
@torch.no_grad()
def answer_logprob(m, prompt, answer):
    ids = tokenizer(prompt + answer, return_tensors="pt").input_ids.to(m.device)
    n_prompt = len(tokenizer(prompt).input_ids)
    logp = torch.log_softmax(m(ids).logits[0, :-1].float(), dim=-1)
    targets = ids[0, 1:]
    return logp[torch.arange(len(targets)), targets][n_prompt - 1:].sum().item()


rows = []
for pair in pairs[:4]:
    with tuned.disable_adapter():
        ref_c, ref_r = answer_logprob(tuned, pair["prompt"], pair["chosen"]), answer_logprob(tuned, pair["prompt"], pair["rejected"])
    pol_c, pol_r = answer_logprob(tuned, pair["prompt"], pair["chosen"]), answer_logprob(tuned, pair["prompt"], pair["rejected"])
    rows.append({"chosen: log ratio": pol_c - ref_c, "rejected: log ratio": pol_r - ref_r,
                 "margin (x beta)": BETA * ((pol_c - ref_c) - (pol_r - ref_r))})
pd.DataFrame(rows).round(2)

A positive margin means the trained model prefers the chosen answer more than the frozen copy did. Note that both answers can lose probability: DPO only needs the rejected one to lose more.

## 7. DPO or plain SFT on the chosen answers?

The chosen answers alone could be used for supervised fine-tuning. Train a fresh adapter that way, on the same questions, and compare. SFT copies the short answers; DPO also learns what to move away from.

In [ ]:
del trainer
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
sft_rows = [{"prompt": p["prompt"], "completion": p["chosen"]} for p in pairs]
sft_args = SFTConfig(output_dir="sft-chosen", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
                     gradient_accumulation_steps=GRAD_ACCUM, learning_rate=2e-4, logging_steps=5, max_length=512,
                     save_strategy="no", report_to="none", seed=SEED, fp16=FP16, bf16=BF16, use_cpu=DEVICE == "cpu")
sft_trainer = SFTTrainer(model=base, args=sft_args, train_dataset=Dataset.from_list(sft_rows),
                         processing_class=tokenizer, peft_config=lora_config)
sft_trainer.train()
sft_score, sft_outs = score(sft_trainer.model.eval(), name="SFT on chosen answers")
pd.concat([before, asked, after, sft_score], axis=1).T

## Summary

- Preference data pairs a better and a worse answer to the same prompt; public sets such as UltraFeedback rate answers with a strong model.
- A model's own answers, produced with and without an instruction, make pairs that teach the instructed behaviour without the instruction.
- `DPOTrainer` needs only the pairs and a LoRA configuration; the base model with the adapter off is the frozen copy.
- The logged reward accuracy and margin show training progress; the real check is the score on held-out questions.
- DPO's margin compares log-probabilities with a frozen copy; SFT on the chosen answers is a simpler baseline worth running next to it.

## Exercises

1. Set `LEARNING_RATE` to 5e-5 and train again. What happens to the answer length, and to the correctness column? Then try `BETA` at 0.5 with that learning rate.
2. Use the model's answers to the same questions in Arabic (translate a few questions) to see whether the brevity carries over to another language.
3. Make pairs for a different behaviour, for example answering in a numbered list, and train again.
4. Train on UltraFeedback pairs instead (use `N_TRAIN` of them), and compare the answers before and after.

## Further Reading

- [Rafailov et al., 2023. Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)
- [Ouyang et al., 2022. Training Language Models to Follow Instructions with Human Feedback](https://arxiv.org/abs/2203.02155)
- [Cui et al., 2023. UltraFeedback: Boosting Language Models with Scaled AI Feedback](https://arxiv.org/abs/2310.01377)
- [Hugging Face TRL: DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)

### Contributed by: Yazen Shaebi